# FLAG 2027 - Notebook 17: ReDimNet2-B6 + voice age/gender (step A of ARCHITECTURE_v2)

The voice side still pays (ResNet293 gave -0.9). ReDimNet2-B6 is ~2x stronger on VoxCeleb1-O (0.23% vs 0.53%) and was trained on 110k multilingual speakers. The audeering wav2vec2 model gives voice **age** and gender (allowed frozen by the organizers' FAQ).

1. Unpack challenge train/dev + MAV-Celeb v1/v3 + the v2/v1-held-out extras, in exactly the row order of our saved features (checked by recomputing ECAPA on a sample).
2. Extract `redim` (192-d) and `audeer` (1024-d pooled + 4 scalars: age, child/female/male).
3. Compare feature combos in the challenge + all-MAV setting with fixed prototypes (reference recomputed here).

About 1.5-2 h. Resumable. Paste the **SUMMARY** to Claude.

In [ ]:
# Cell 1 - Drive, GPU, packages
from google.colab import drive
drive.mount('/content/drive')
!pip -q install speechbrain
import torch, shutil
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', dev, '| disk free %.0f GB' % (shutil.disk_usage('/content')[2] / 1e9))

In [ ]:
# Cell 2 - file lists in the SAME order as our feature files (challenge splits, MAV v1/v3, MAV extra)
import os, re, json, time, zipfile, copy
import numpy as np, pandas as pd
import librosa, soundfile as sf
from tqdm.auto import tqdm

BASE = '/content/drive/MyDrive'
PROJECT = os.path.join(BASE, 'FLAG2027')
FEATDIR = os.path.join(PROJECT, 'features')
RESDIR = os.path.join(PROJECT, 'results')
CACHE = os.path.join(PROJECT, 'models_cache')
ZIP_DIR = json.load(open(os.path.join(PROJECT, 'paths.json')))['ZIP_DIR']
MAV_ZIPS, MAV_LISTS = os.path.join(BASE, 'dataset'), os.path.join(BASE, 'face_voice_association_splits')
SUMMARY = []
def note(msg):
    print(msg); SUMMARY.append(msg)

# challenge (wav + lists only)
FULL = '/content/flag_data'; os.makedirs(FULL, exist_ok=True)
for z in ['train_set.zip', 'dev_set.zip']:
    if not os.path.exists(os.path.join(FULL, z + '.done')):
        with zipfile.ZipFile(os.path.join(ZIP_DIR, z)) as zf:
            zf.extractall(FULL, members=[m for m in zf.namelist() if m.endswith(('.wav', '.txt', '.csv'))])
        open(os.path.join(FULL, z + '.done'), 'w').close()
TXT = [os.path.join(r, f) for r, _, fs in os.walk(FULL) for f in fs if f.endswith(('.txt', '.csv'))]
def find_one(*parts):
    hits = [p for p in TXT if all(x in p for x in parts)]
    assert len(hits) == 1, 'looking for %s -> found %s' % (parts, hits)
    return hits[0]
def resolve(rel, list_path):
    q = os.path.dirname(list_path)
    while q.startswith(FULL):
        c = os.path.join(q, rel)
        if os.path.exists(c):
            return c
        q = os.path.dirname(q)
SPLIT_NAMES = ['train', 'no_gender_English', 'no_gender_Bangla', 'gender_English', 'gender_Bangla']
FILES = {}
lp = find_one('train_English.txt'); FILES['train'] = [resolve(l.split()[2], lp) for l in open(lp) if l.strip()]
for s in SPLIT_NAMES[1:]:
    track, lang_ = s.rsplit('_', 1); lp = find_one('/' + track + '/', lang_ + '_test.txt')
    FILES[s] = [resolve(l.split()[1], lp) for l in open(lp) if l.strip()]

# MAV v1/v3 train (as Notebook 06: keep a row only if voice AND face exist)
def zip_index(zf):
    m = {}
    for n in zf.namelist():
        for key in ('voices/', 'faces/'):
            i = n.find(key)
            if i >= 0 and not n.endswith('/'):
                m[n[i:]] = n; break
    return m
MAVD = '/content/mav'; mav_paths, mspk = [], []
for ver, zname, lists in [('v1', 'mavceleb_v1_train.zip', ['English_train.txt', 'Urdu_train.txt']),
                          ('v3', 'mavceleb_v3_tran.zip', ['English_train.txt', 'German_train.txt'])]:
    flag = os.path.join(MAVD, ver + '.voices.done'); need = not os.path.exists(flag)
    zf = zipfile.ZipFile(os.path.join(MAV_ZIPS, zname)); zi = zip_index(zf)
    for lf in lists:
        for l in open(os.path.join(MAV_LISTS, ver, lf)):
            t = l.split()
            if len(t) < 2:
                continue
            vm, fm = zi.get('voices/' + t[0]), zi.get('faces/' + t[1])
            if vm is None or fm is None:
                continue
            if need:
                zf.extract(vm, os.path.join(MAVD, ver))
            mav_paths.append(os.path.join(MAVD, ver, vm)); mspk.append(ver + '_' + t[0].split('/')[0])
    zf.close()
    if need:
        open(flag, 'w').close()
MAV = dict(np.load(os.path.join(FEATDIR, 'mavceleb_v1v3.npz')))
assert list(MAV['spk']) == mspk, 'MAV v1/v3 row order differs - tell Claude'
FILES['mav'] = mav_paths

# MAV extra (v2 all + v1 held-out), rebuilt with the same deterministic selection as Notebook 14
EXTRA = dict(np.load(os.path.join(FEATDIR, 'mavceleb_extra.npz')))
MAX_PER_ID = 120
VPAT = re.compile(r'voices/(id\d+)/([^/]+)/([^/]+)/([^/]+\.wav)$'); FPAT = re.compile(r'faces/(id\d+)/([^/]+)/([^/]+)/([^/]+\.jpg)$')
v1_used = set()
for lf in ['English_train.txt', 'Urdu_train.txt']:
    for l in open(os.path.join(MAV_LISTS, 'v1', lf)):
        t = l.split()
        if t:
            v1_used.add(t[0].split('/')[0])
rows = []
for ver, zname, skip_ids in [('v2', 'mavceleb_v2_complete.zip', set()), ('v1', 'mavceleb_v1_complete.zip', v1_used)]:
    D = '/content/mavx/' + ver; flag = os.path.join(D, 'voices.done'); need = not os.path.exists(flag)
    zf = zipfile.ZipFile(os.path.join(MAV_ZIPS, zname))
    voices, faces = {}, {}
    for n in zf.namelist():
        mv = VPAT.search(n)
        if mv:
            voices.setdefault(mv.group(1), []).append((mv.group(2), mv.group(3), n)); continue
        mf = FPAT.search(n)
        if mf:
            faces.setdefault((mf.group(1), mf.group(2), mf.group(3)), []).append(n)
    ids_ = sorted(i for i in voices if i not in skip_ids)
    rng = np.random.default_rng(0)
    for pid in ids_:
        vs = voices[pid]; vs = [vs[j] for j in rng.permutation(len(vs))[:MAX_PER_ID]]
        for lang_, vid, vn in vs:
            fr = faces.get((pid, lang_, vid)) or [f for (p, l, v), fl in faces.items() if p == pid for f in fl]
            if not fr:
                continue
            _ = fr[rng.integers(len(fr))]          # consume the same random draw as Notebook 14
            if need:
                zf.extract(vn, D)
            rows.append((ver + '_' + pid, os.path.join(D, vn)))
    zf.close()
    if need:
        open(flag, 'w').close()
# Notebook 14 then removed Bengali-detected clips; we recover that filter by matching the saved speaker sequence
spk_all = np.array([r[0] for r in rows]); keep = np.ones(len(rows), bool)
if len(rows) != len(EXTRA['spk']):
    # drop rows greedily so that the speaker sequence matches (27 Bengali clips were removed)
    j = 0; keep[:] = False
    for i in range(len(rows)):
        if j < len(EXTRA['spk']) and spk_all[i] == EXTRA['spk'][j]:
            keep[i] = True; j += 1
    assert j == len(EXTRA['spk']), 'could not align MAV extra rows - tell Claude'
FILES['extra'] = [r[1] for r, k in zip(rows, keep) if k]
note('Files: ' + ', '.join('%s %d' % (k, len(v)) for k, v in FILES.items()))

In [ ]:
# Cell 3 - alignment check (recompute official ECAPA on 200 rows of each extracted set and compare with saved)
try:
    from speechbrain.inference.interfaces import Pretrained
except ImportError:
    from speechbrain.pretrained.interfaces import Pretrained
class Encoder(Pretrained):
    MODULES_NEEDED = ['compute_features', 'mean_var_norm', 'embedding_model']
    def encode_batch(self, wavs, wav_lens=None):
        if wav_lens is None:
            wav_lens = torch.ones(wavs.shape[0], device=self.device)
        wavs, wav_lens = wavs.to(self.device).float(), wav_lens.to(self.device)
        return self.mods.embedding_model(self.mods.mean_var_norm(self.mods.compute_features(wavs), wav_lens), wav_lens)
ECAPA = Encoder.from_hparams(source='yangwang825/ecapa-tdnn-vox2', savedir=os.path.join(CACHE, 'ecapa_vox2'), run_opts={'device': dev}); ECAPA.eval()
def ecapa_one(p):
    w = torch.from_numpy(librosa.load(p, sr=16000, mono=True)[0]).unsqueeze(0)
    with torch.no_grad():
        return ECAPA.encode_batch(w).squeeze().cpu().numpy()
def read_feat(path, dim):
    df = pd.read_csv(path, header=None)
    return df.iloc[:, :dim].values.astype(np.float32), (df.iloc[:, dim].values if df.shape[1] > dim else None)
Xv, y = read_feat(find_one('train_English_voices.csv'), 192)
SAVED = {'train': Xv, 'mav': MAV['ecapa'], 'extra': EXTRA['ecapa']}
rng = np.random.default_rng(1)
for k, saved in SAVED.items():
    idx = rng.choice(len(saved), 200, replace=False)
    new = np.stack([ecapa_one(FILES[k][i]) for i in idx]); old = saved[idx]
    cos = (new * old).sum(1) / (np.linalg.norm(new, axis=1) * np.linalg.norm(old, axis=1) + 1e-9)
    note('alignment %-6s median cosine %.4f' % (k, np.median(cos)))
    assert np.median(cos) > 0.98, 'rows of %s do not line up - stop and tell Claude' % k

In [ ]:
# Cell 4 - encoders: ReDimNet2-B6 (with fallbacks) and audeering age-gender   [v2: two bug fixes]
import torch.nn as nn
def load_redimnet():
    tries = [('PalabraAI/redimnet2', dict(model_name='b6', train_type='lm', dataset='vb2+vox2_v0', pretrained=True), 'redimnet2'),
             ('PalabraAI/redimnet2', dict(model_name='b6', train_type='lm', dataset='vox2', pretrained=True), 'redimnet2'),
             ('IDRnD/ReDimNet', dict(model_name='b6', train_type='ft_lm', dataset='vox2'), 'ReDimNet')]
    for repo, kw, entry in tries:
        try:
            m = torch.hub.load(repo, entry, **kw).to(dev).eval()
            note('ReDimNet loaded: %s %s' % (repo, kw)); return m
        except Exception as e:
            print('  failed', repo, kw, '->', repr(e)[:160])
    raise RuntimeError('no ReDimNet variant could be loaded')

def enc_redim(paths, bs=16, max_sec=10):
    net = load_redimnet()
    order = np.argsort([sf.info(p).duration for p in paths])
    emb = {}
    for i in tqdm(range(0, len(order), bs), desc='redimnet'):
        idx = order[i:i + bs]
        wavs = [librosa.load(paths[j], sr=16000, mono=True)[0][:16000 * max_sec] for j in idx]
        L = max(len(w) for w in wavs)
        b = torch.zeros(len(wavs), L)
        for k_, w in enumerate(wavs):
            b[k_] = torch.from_numpy(np.tile(w, int(np.ceil(L / max(len(w), 1))))[:L])   # repeat-pad to L (fix: full row)
        with torch.no_grad():
            e = net(b.to(dev))
        e = e[0] if isinstance(e, (list, tuple)) else e
        for k_, j in enumerate(idx):
            emb[int(j)] = e[k_].float().cpu().numpy().reshape(-1)
    return np.stack([emb[j] for j in range(len(paths))])

from transformers import Wav2Vec2Processor, Wav2Vec2Model, Wav2Vec2PreTrainedModel
from huggingface_hub import hf_hub_download
class ModelHead(nn.Module):
    def __init__(self, hidden, num_labels, drop):
        super().__init__()
        self.dense = nn.Linear(hidden, hidden); self.dropout = nn.Dropout(drop); self.out_proj = nn.Linear(hidden, num_labels)
    def forward(self, x):
        return self.out_proj(self.dropout(torch.tanh(self.dense(self.dropout(x)))))
class AgeGenderModel(Wav2Vec2PreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.wav2vec2 = Wav2Vec2Model(config)
        self.age = ModelHead(config.hidden_size, 1, config.final_dropout); self.gender = ModelHead(config.hidden_size, 3, config.final_dropout)
        self.post_init()                                   # fix: new transformers needs post_init, not init_weights
    def forward(self, input_values):
        h = self.wav2vec2(input_values)[0].mean(1)
        return h, self.age(h), torch.softmax(self.gender(h), dim=1)

def load_audeer():
    name = 'audeering/wav2vec2-large-robust-24-ft-age-gender'
    proc = Wav2Vec2Processor.from_pretrained(name)
    try:
        model = AgeGenderModel.from_pretrained(name)
    except Exception as e:                                 # fallback: base model + heads loaded by hand
        print('  from_pretrained failed (%r) -> manual load' % (repr(e)[:120],))
        base = Wav2Vec2Model.from_pretrained(name)
        model = AgeGenderModel(base.config); model.wav2vec2 = base
        try:
            sd_path = hf_hub_download(name, 'model.safetensors'); from safetensors.torch import load_file; sd = load_file(sd_path)
        except Exception:
            sd = torch.load(hf_hub_download(name, 'pytorch_model.bin'), map_location='cpu')
        heads = {k: v for k, v in sd.items() if k.startswith(('age.', 'gender.'))}
        print('  head weights loaded:', model.load_state_dict(heads, strict=False).unexpected_keys == [])
    return proc, model.to(dev).eval().half()

def enc_audeer(paths, bs=16, max_sec=8):
    proc, model = load_audeer()
    H, S = [], []
    for i in tqdm(range(0, len(paths), bs), desc='audeering age-gender'):
        wavs = [librosa.load(p, sr=16000, mono=True)[0][:16000 * max_sec] for p in paths[i:i + bs]]
        x = proc(wavs, sampling_rate=16000, padding=True, return_tensors='pt')['input_values'].to(dev).half()
        with torch.no_grad():
            h, a, g = model(x)
        H.append(h.float().cpu().numpy()); S.append(torch.cat([a, g], 1).float().cpu().numpy())
    return np.concatenate(H), np.concatenate(S)
print('encoders defined')

In [ ]:
# Cell 5 - extract + save (skipped if present)
ORDER = SPLIT_NAMES + ['mav', 'extra']
allp = [p for s in ORDER for p in FILES[s]]
def split_back(emb):
    out, i = {}, 0
    for s in ORDER:
        n = len(FILES[s]); out[s] = emb[i:i + n].astype(np.float32); i += n
    return out
p_r = os.path.join(FEATDIR, 'redim.npz')
if not os.path.exists(p_r):
    t0 = time.time()
    try:
        np.savez(p_r, **split_back(enc_redim(allp))); note('redim extracted for %d files in %.1f min' % (len(allp), (time.time() - t0) / 60))
    except Exception as e:
        note('ENCODER redim FAILED: %r' % e)
p_a = os.path.join(FEATDIR, 'audeer.npz')
if not os.path.exists(p_a):
    t0 = time.time()
    try:
        H, S = enc_audeer(allp)
        d = split_back(H); d.update({k + '_ag': v for k, v in split_back(S).items()})
        np.savez(p_a, **d); note('audeer extracted for %d files in %.1f min' % (len(allp), (time.time() - t0) / 60))
    except Exception as e:
        note('ENCODER audeer FAILED: %r' % e)

In [ ]:
# Cell 6 - training setup (challenge + ALL MAV, per-dataset norm, fixed prototypes = P1 recipe without VoxCeleb)
import torch.nn.functional as Fn
from sklearn.metrics import roc_curve
FEAT = {'ecapa': {'train': Xv, 'mav': MAV['ecapa'], 'extra': EXTRA['ecapa']}}
for s in SPLIT_NAMES[1:]:
    track, lang_ = s.rsplit('_', 1)
    FEAT['ecapa'][s] = read_feat(find_one('/' + track + '/', lang_ + '_test_voices.csv'), 192)[0]
for name in ['clip', 'wavlm_sv', 'r293']:
    d = np.load(os.path.join(FEATDIR, name + '.npz')); FEAT[name] = {s: d[s] for s in d.files}
FEAT['clip']['mav'], FEAT['clip']['extra'] = MAV['clip'], EXTRA['clip']
FEAT['wavlm_sv']['mav'], FEAT['wavlm_sv']['extra'] = MAV['wavlm_sv'], EXTRA['wavlm_sv']
FEAT['r293']['extra'] = EXTRA['r293']
for name in ['redim', 'audeer']:
    p = os.path.join(FEATDIR, name + '.npz')
    if os.path.exists(p):
        d = np.load(p); FEAT[name] = {s: d[s] for s in ORDER}
        if name == 'audeer':
            FEAT['audeer_ag'] = {s: d[s + '_ag'] for s in ORDER}
y = y.astype(int)
train_lines = [l.split() for l in open(find_one('train_English.txt')) if l.strip()]
cls2spk = {int(l[5]): l[4] for l in train_lines}
meta = pd.read_csv(os.path.join(ZIP_DIR, 'meta_file_train_set.csv'), header=None, names=['spk', 'g'])
spk2g = dict(zip(meta.spk.astype(str).str.strip(), meta.g.astype(str).str.strip().str.lower()))
ids = np.unique(y); id_gender = {c: spk2g[cls2spk[c]] for c in ids}
N_CH = len(y)
ext_spk = list(MAV['spk']) + list(EXTRA['spk'])
ext_lab = {s: 1000 + i for i, s in enumerate(sorted(set(ext_spk)))}
Y_ALL = np.r_[y, np.array([ext_lab[s] for s in ext_spk])]
D_ALL = np.r_[np.zeros(N_CH, int), np.ones(len(ext_spk), int)]
EXT = N_CH + np.arange(len(ext_spk))
def l2(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
def build(names, split):
    blocks = []
    for n in names:
        X = FEAT[n][split]
        blocks.append(X if n == 'audeer_ag' else l2(X))        # the 4 age/gender scalars are used raw
    return np.concatenate(blocks, 1).astype(np.float32)
def build_train(names):
    return np.concatenate([build(names, 'train'), build(names, 'mav'), build(names, 'extra')])

def eer(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores); fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr)); return (fpr[i] + fnr[i]) / 2 * 100
def make_pairs(idx, n=2000, same_gender_neg=False, seed=0):
    rng = np.random.default_rng(seed)
    spk_ = np.unique(y[idx]); rows_ = {s: idx[y[idx] == s] for s in spk_}
    V, F, L = [], [], []
    for _ in range(n):
        s = rng.choice(spk_); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[s])); L.append(1)
        s = rng.choice(spk_)
        cand = [t for t in spk_ if t != s and (not same_gender_neg or id_gender[t] == id_gender[s])] or [t for t in spk_ if t != s]
        t = rng.choice(cand); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[t])); L.append(0)
    return np.array(V), np.array(F), np.array(L)
K = 5
rng = np.random.default_rng(42)
folds = [[] for _ in range(K)]
for gg in sorted(set(id_gender.values())):
    members = [c for c in ids if id_gender[c] == gg]; rng.shuffle(members)
    for i, c in enumerate(members):
        folds[i % K].append(c)
FOLD_DATA = []
for k in range(K):
    val_ids = set(folds[k])
    va_idx = np.where([c in val_ids for c in y])[0]; tr_idx = np.where([c not in val_ids for c in y])[0]
    FOLD_DATA.append((tr_idx, [make_pairs(va_idx, seed=k), make_pairs(va_idx, same_gender_neg=True, seed=100 + k)]))

_PROTO = {}
def max_sep_prototypes(n):
    if n in _PROTO:
        return _PROTO[n]
    P = np.array([[1.0, -1.0]])
    for k in range(2, n):
        Q = np.zeros((k, k + 1)); Q[0, 0] = 1.0; Q[0, 1:] = -1.0 / k; Q[1:, 1:] = np.sqrt(1 - 1.0 / k ** 2) * P
        P = Q
    _PROTO[n] = torch.tensor(P.T, dtype=torch.float32); return _PROTO[n]
class Branch(nn.Module):
    def __init__(self, d_in, d=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Dropout(0.5))
    def forward(self, x):
        return Fn.normalize(self.net(x), dim=1)
class FOP(nn.Module):
    def __init__(self, n_class, d_face, d_voice, d=128):
        super().__init__()
        self.face, self.voice = Branch(d_face, d), Branch(d_voice, d)
        self.att = nn.Sequential(nn.Linear(2 * d, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Linear(d, d))
        self.cls = nn.Linear(d, n_class - 1); self.register_buffer('P', max_sep_prototypes(n_class))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight); nn.init.constant_(m.bias, 0.01)
    def embed(self, xf, xv):
        f, v = self.face(xf), self.voice(xv)
        return torch.tanh(f), torch.tanh(v), f, v
    def forward(self, xf, xv):
        ft, vt, f, v = self.embed(xf, xv)
        a = torch.sigmoid(self.att(torch.cat([f, v], 1)))
        fused = ft * a + (1 - a) * vt
        return fused, self.cls(fused) @ self.P.t()
def opl_loss(feat, lab):
    feat = Fn.normalize(feat, dim=1); dot = feat @ feat.t()
    same = (lab[:, None] == lab[None, :]).float()
    pos = same - torch.eye(len(lab), device=feat.device); neg = 1 - same
    return (1 - (pos * dot).sum() / (pos.sum() + 1e-6)) + 0.7 * (neg * dot.abs()).sum() / (neg.sum() + 1e-6)
@torch.no_grad()
def same_score(model, xf, xv):
    model.eval(); mf, sf_, mv, sv = model.norm
    ft, vt, _, _ = model.embed((xf - mf) / sf_, (xv - mv) / sv)
    return -(ft - vt).norm(dim=1).cpu().numpy()
print('setup ready | features:', list(FEAT))

In [ ]:
# Cell 7 - compare feature combos (RESUMABLE)
SEEDS = [1, 2, 3]
P_ID, K_CLIP, ALPHA, LR, MAX_EP, PATIENCE, STEPS = 16, 8, 2.0, 1e-4, 100, 15, 60
def dnorm(F_raw, V_raw, tr_idx):
    F, V = F_raw.copy(), V_raw.copy(); ch = None
    for d in np.unique(D_ALL[tr_idx]):
        trd = tr_idx[D_ALL[tr_idx] == d]; alld = np.where(D_ALL == d)[0]
        st = (F_raw[trd].mean(0), F_raw[trd].std(0) + 1e-6, V_raw[trd].mean(0), V_raw[trd].std(0) + 1e-6)
        F[alld] = (F_raw[alld] - st[0]) / st[1]; V[alld] = (V_raw[alld] - st[2]) / st[3]
        if d == 0:
            ch = st
    return F, V, ch
def train_combo(F_raw, V_raw, tr_idx, vsets=None, seed=1, max_epochs=MAX_EP):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    all_tr = np.concatenate([tr_idx, EXT])
    F, V, ch = dnorm(F_raw, V_raw, all_tr)
    XF = torch.tensor(F, device=dev); XV = torch.tensor(V, device=dev)
    cl = np.unique(Y_ALL[all_tr]); rm = {c: i for i, c in enumerate(cl)}
    ytr_np = np.array([rm[c] for c in Y_ALL[all_tr]]); ytr = torch.tensor(ytr_np, device=dev)
    pos_by_cls = [np.where(ytr_np == c)[0] for c in range(len(cl))]
    tr = torch.tensor(all_tr, device=dev)
    model = FOP(len(cl), XF.shape[1], XV.shape[1]).to(dev)
    model.norm = (XF[tr].mean(0), XF[tr].std(0) + 1e-6, XV[tr].mean(0), XV[tr].std(0) + 1e-6)
    mf, sf_, mv, sv = model.norm
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.01)
    best, best_ep, best_state, wait = 1e9, max_epochs, None, 0
    for ep in range(1, max_epochs + 1):
        model.train()
        for _ in range(STEPS):
            chosen = rng.choice(len(cl), P_ID, replace=False)
            j = torch.tensor(np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen]), device=dev)
            fused, logits = model((XF[tr[j]] - mf) / sf_, (XV[tr[j]] - mv) / sv)
            loss = Fn.cross_entropy(logits, ytr[j]) + ALPHA * opl_loss(fused, ytr[j])
            opt.zero_grad(); loss.backward(); opt.step()
        if vsets is None:
            continue
        score = np.mean([eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets])
        if score < best - 0.05:
            best, best_ep, best_state, wait = score, ep, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_ep, XF, XV, ch

COMBOS = {
    'R0_ref_ecapa+wavlm+r293':      (['clip'], ['ecapa', 'wavlm_sv', 'r293']),
    'R1_+redim':                    (['clip'], ['ecapa', 'wavlm_sv', 'r293', 'redim']),
    'R2_redim_replaces_r293':       (['clip'], ['ecapa', 'wavlm_sv', 'redim']),
    'R3_+redim+age4':               (['clip'], ['ecapa', 'wavlm_sv', 'r293', 'redim', 'audeer_ag']),
    'R4_+redim+audeer1024':         (['clip'], ['ecapa', 'wavlm_sv', 'r293', 'redim', 'audeer']),
}
COMBOS = {k: v for k, v in COMBOS.items() if all(n in FEAT for n in v[0] + v[1])}
print('Combos:', list(COMBOS))
CSV = os.path.join(RESDIR, 'nb17_cv.csv')
done = pd.read_csv(CSV) if os.path.exists(CSV) else pd.DataFrame(columns=['combo', 'seed', 'fold'])
t0 = time.time()
for name, (fn_, vn_) in COMBOS.items():
    F_raw, V_raw = build_train(fn_), build_train(vn_)
    for seed in SEEDS:
        for k in range(K):
            if ((done.combo == name) & (done.seed == seed) & (done.fold == k)).any():
                continue
            tr_idx, vsets = FOLD_DATA[k]
            model, best_ep, XF, XV, _ = train_combo(F_raw, V_raw, tr_idx, vsets, seed=seed)
            e_std, e_gen = [eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets]
            row = pd.DataFrame([dict(combo=name, seed=seed, fold=k, best_ep=best_ep, std=e_std, gender=e_gen)])
            row.to_csv(CSV, mode='a', header=not os.path.exists(CSV), index=False)
            done = pd.concat([done, row])
            print('%-26s seed %d fold %d | ep %3d | std %.2f | gender %.2f | %.0f min' % (name, seed, k, best_ep, e_std, e_gen, (time.time() - t0) / 60))
R = pd.read_csv(CSV); R['overall'] = (R['std'] + R['gender']) / 2
tab = R.groupby(['combo', 'seed'])[['std', 'gender', 'overall']].mean().groupby('combo').mean()
tab = tab.loc[[c for c in COMBOS if c in tab.index]].round(2)
note('NEW VOICE ENCODERS (challenge + all MAV, prototypes; 5-fold x 3 seeds)')
note(tab.to_string())
BEST = tab['overall'].idxmin()
note('BEST: %s (%.2f) vs reference %.2f' % (BEST, tab.loc[BEST, 'overall'], tab.loc['R0_ref_ecapa+wavlm+r293', 'overall']))

In [ ]:
# Cell 8 - submission ZIP for the best combo (challenge + all MAV, no VoxCeleb) + saved dev scores
LANG_FILE = {'English': 'sub_score_v4_English_heard.txt', 'Bangla': 'sub_score_v4_Bangla_unheard.txt'}
SUBDIR = os.path.join(PROJECT, 'submissions'); os.makedirs(SUBDIR, exist_ok=True)
fn_, vn_ = COMBOS[BEST]
F_raw, V_raw = build_train(fn_), build_train(vn_)
ep = int(np.median(R[R.combo == BEST]['best_ep']))
finals = [train_combo(F_raw, V_raw, np.arange(N_CH), None, seed=s, max_epochs=ep)[0] for s in SEEDS]
_, _, _, _, ch = train_combo(F_raw, V_raw, np.arange(N_CH), None, seed=1, max_epochs=1)
sc = {}
for track in ['no_gender', 'gender']:
    for lang_ in LANG_FILE:
        split = track + '_' + lang_
        fx = ((build(fn_, split) - ch[0]) / ch[1]).astype(np.float32); vx = ((build(vn_, split) - ch[2]) / ch[3]).astype(np.float32)
        xf = torch.tensor(fx, device=dev); xv = torch.tensor(vx, device=dev)
        sc[split] = np.mean([same_score(m, xf, xv) for m in finals], axis=0)
tag = BEST.split('_')[0]
OUT = '/content/sub_17_' + tag
for track in ['no_gender', 'gender']:
    os.makedirs(os.path.join(OUT, track), exist_ok=True)
    for lang_, fname in LANG_FILE.items():
        split = track + '_' + lang_
        pair_ids = [l.split()[0] for l in open(find_one('/' + track + '/', lang_ + '_test.txt')) if l.strip()]
        assert len(pair_ids) == len(sc[split]) and np.isfinite(sc[split]).all()
        with open(os.path.join(OUT, track, fname), 'w') as f:
            for pid, v in zip(pair_ids, -sc[split]):
                f.write('%s %.6f\n' % (pid, v))
zp = os.path.join(SUBDIR, '17_%s_noVox.zip' % tag)
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as zf:
    for track in ['no_gender', 'gender']:
        for fname in LANG_FILE.values():
            zf.write(os.path.join(OUT, track, fname), arcname=track + '/' + fname)
np.savez(os.path.join(RESDIR, 'devscores_17_%s.npz' % tag), **sc)
note('Submission ZIP saved: ' + zp[len(BASE):])
print('=' * 70)
print('SUMMARY  (copy everything between the lines and paste to Claude)')
print('=' * 70)
for s in SUMMARY:
    print(s)
print('=' * 70)